In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Used Gemini AI to produce code

In [ ]:
# linear_regression_house_prices.py
# This script trains a Linear Regression model on the Ames Housing dataset
# from the Kaggle "House Prices: Advanced Regression Techniques" competition.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Data Loading and Initial Exploration ---

# Define the correct file paths for the Kaggle environment
TRAIN_PATH = '../input/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '../input/house-prices-advanced-regression-techniques/test.csv'

# Load the training and testing datasets
try:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
except FileNotFoundError:
    print(f"Ensure '{TRAIN_PATH}' and '{TEST_PATH}' are correct.")
    # You can download the data from: https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data
    # If running locally, change the paths to 'train.csv' and 'test.csv'
    raise


# Store the 'Id' column from the test set for the final submission file
# and drop it from the training set as it's not a feature.
test_ids = test_df['Id']
train_df = train_df.drop('Id', axis=1)
test_df = test_df.drop('Id', axis=1)

print(f"Training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")


# --- 2. Preprocessing and Feature Engineering ---

# The evaluation metric is RMSLE (Root Mean Squared Logarithmic Error).
# To optimize for this, we should predict the logarithm of the SalePrice.
# We use np.log1p which is log(1+x) to handle potential zero values.
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

# Combine train and test data for consistent preprocessing
all_data = pd.concat((train_df.drop(['SalePrice'], axis=1), test_df))
print(f"Combined data shape: {all_data.shape}")

# Identify numerical and categorical features
numerical_features = all_data.select_dtypes(include=np.number).columns
categorical_features = all_data.select_dtypes(exclude=np.number).columns

# --- 2a. Handle Missing Values ---

# For categorical features, 'NA' often means 'None' (e.g., no pool).
# We'll fill these with the string 'None'.
for col in categorical_features:
    all_data[col] = all_data[col].fillna('None')

# For numerical features, we'll impute missing values with the median.
# The median is more robust to outliers than the mean.
for col in numerical_features:
    all_data[col] = all_data[col].fillna(all_data[col].median())

# --- 2b. Feature Transformation ---

# One-hot encode categorical features. This converts categories into binary columns.
all_data = pd.get_dummies(all_data, drop_first=True)

# Separate the combined data back into training and testing sets
X = all_data[:train_df.shape[0]]
X_test = all_data[train_df.shape[0]:]
y = train_df['SalePrice']

print(f"Processed training features shape: {X.shape}")
print(f"Processed testing features shape: {X_test.shape}")


# --- 3. Model Training ---

# Initialize the Linear Regression model
lr_model = LinearRegression()

# Train the model on the entire training dataset
lr_model.fit(X, y)

print("\nLinear Regression model trained successfully.")

# Calculate RMSE on the training data (just for a rough idea of fit)
train_predictions_log = lr_model.predict(X)
train_rmse = np.sqrt(mean_squared_error(y, train_predictions_log))
print(f"RMSE on training data (log scale): {train_rmse:.4f}")


# --- 4. Generate Predictions for Submission ---

# Make predictions on the preprocessed test data
test_predictions_log = lr_model.predict(X_test)

# The model predicts the log of the price, so we need to convert it back.
# We use np.expm1 which is exp(x) - 1, the inverse of np.log1p.
final_predictions = np.expm1(test_predictions_log)

# Ensure no negative predictions (which can happen with linear models)
final_predictions[final_predictions < 0] = 0


# --- 5. Create Submission File ---

# Create a DataFrame in the format required by Kaggle
submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Save the submission file
submission_df.to_csv('submission_linear_regression.csv', index=False)

print("\nSubmission file 'submission_linear_regression.csv' created successfully.")
print("You can now upload this file to the Kaggle competition.")
print(submission_df.head())

In [ ]:
# neural_net_house_prices.py
# This script trains a Multi-layer Perceptron (Neural Net) model on the Ames Housing dataset
# from the Kaggle "House Prices: Advanced Regression Techniques" competition.

import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# --- 1. Data Loading and Initial Exploration ---

# Define the correct file paths for the Kaggle environment
TRAIN_PATH = '../input/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '../input/house-prices-advanced-regression-techniques/test.csv'

# Load the training and testing datasets
try:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
except FileNotFoundError:
    print(f"Ensure '{TRAIN_PATH}' and '{TEST_PATH}' are correct.")
    # If running locally, change the paths to 'train.csv' and 'test.csv'
    raise

# Store the 'Id' column from the test set for the final submission file
test_ids = test_df['Id']
# Drop Id columns to avoid treating them as features
train_df = train_df.drop('Id', axis=1)
test_df = test_df.drop('Id', axis=1)

print(f"Original training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

# --- 2. Preprocessing and Feature Engineering ---

# --- 2a. Handle Outliers ---
# The dataset has two known outliers with large GrLivArea but low SalePrice.
# Removing them is a standard practice for this competition.
train_df = train_df.drop(train_df[(train_df['GrLivArea']>4000) & (train_df['SalePrice']<300000)].index)
print(f"Training data shape after removing outliers: {train_df.shape}")


# --- 2b. Target Variable Transformation ---
# The evaluation metric is RMSLE. To optimize for this, we predict the logarithm of SalePrice.
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])
y = train_df['SalePrice']


# --- 2c. Combine Data and Engineer Features ---
# Combine train and test data for consistent preprocessing
all_data = pd.concat((train_df.drop(['SalePrice'], axis=1), test_df))
print(f"Combined data shape: {all_data.shape}")

# Create more meaningful features
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# --- 2d. Handle Missing Values ---
# Fill categorical NAs with 'None' and numerical NAs with the median.
categorical_features = all_data.select_dtypes(exclude=np.number).columns
for col in categorical_features:
    all_data[col] = all_data[col].fillna('None')

numerical_features = all_data.select_dtypes(include=np.number).columns
for col in numerical_features:
    all_data[col] = all_data[col].fillna(all_data[col].median())

# --- 2e. Handle Skewed Features ---
# Log-transform skewed numerical features to make their distributions more normal.
skewed_feats = all_data[numerical_features].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
skewed_feats = skewed_feats.index
all_data[skewed_feats] = np.log1p(all_data[skewed_feats])

# --- 2f. Final Transformations ---
# One-hot encode categorical features.
all_data = pd.get_dummies(all_data, drop_first=True)

# Separate the combined data back into training and testing sets
X = all_data[:train_df.shape[0]]
X_test = all_data[train_df.shape[0]:]

# --- 2g. Feature Scaling (Crucial for Neural Networks) ---
# Neural Networks perform best when numerical inputs are scaled.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

print(f"Processed training features shape: {X_scaled.shape}")
print(f"Processed testing features shape: {X_test_scaled.shape}")


# --- 3. Model Training ---

# Initialize the Multi-layer Perceptron (Neural Net) Regressor with improvements
# - alpha: Increased L2 regularization to prevent overfitting.
# - early_stopping: Stops training when validation score doesn't improve, preventing overfitting.
# - hidden_layer_sizes: A slightly deeper architecture.
nn_model = MLPRegressor(
    hidden_layer_sizes=(150, 75, 25),
    max_iter=2000,
    random_state=42,
    alpha=0.01, # L2 regularization
    learning_rate='adaptive',
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

# Train the model on the scaled training dataset
nn_model.fit(X_scaled, y)

print("\nNeural Net model trained successfully.")

# Calculate RMSE on the training data
train_predictions_log = nn_model.predict(X_scaled)
train_rmse = np.sqrt(mean_squared_error(y, train_predictions_log))
print(f"RMSE on training data (log scale): {train_rmse:.4f}")


# --- 4. Generate Predictions for Submission ---

# Make predictions on the scaled test data
test_predictions_log = nn_model.predict(X_test_scaled)

# The model predicts the log of the price, so we need to convert it back.
final_predictions = np.expm1(test_predictions_log)

# Ensure no negative predictions
final_predictions[final_predictions < 0] = 0


# --- 5. Create Submission File ---

# Create a DataFrame in the format required by Kaggle
submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Save the submission file
submission_df.to_csv('submission_neural_net_improved.csv', index=False)

print("\nSubmission file 'submission_neural_net_improved.csv' created successfully.")
print("You can now upload this file to the Kaggle competition.")
print(submission_df.head())



In [ ]:
# xgboost_house_prices.py
# This script trains an XGBoost model on the Ames Housing dataset
# from the Kaggle "House Prices: Advanced Regression Techniques" competition.

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# --- 1. Data Loading and Initial Exploration ---

# Define the correct file paths for the Kaggle environment
TRAIN_PATH = '../input/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '../input/house-prices-advanced-regression-techniques/test.csv'

# Load the training and testing datasets
try:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
except FileNotFoundError:
    print(f"Ensure '{TRAIN_PATH}' and '{TEST_PATH}' are correct.")
    # If running locally, change the paths to 'train.csv' and 'test.csv'
    raise

# Store the 'Id' column from the test set for the final submission file
test_ids = test_df['Id']
# Drop Id columns to avoid treating them as features
train_df = train_df.drop('Id', axis=1)
test_df = test_df.drop('Id', axis=1)

print(f"Original training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

# --- 2. Preprocessing and Feature Engineering ---

# --- 2a. Handle Outliers ---
# The dataset has two known outliers with large GrLivArea but low SalePrice.
# Removing them is a standard practice for this competition.
train_df = train_df.drop(train_df[(train_df['GrLivArea']>4000) & (train_df['SalePrice']<300000)].index)
print(f"Training data shape after removing outliers: {train_df.shape}")


# --- 2b. Target Variable Transformation ---
# The evaluation metric is RMSLE. To optimize for this, we predict the logarithm of SalePrice.
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])
y = train_df['SalePrice']


# --- 2c. Combine Data and Engineer Features ---
# Combine train and test data for consistent preprocessing
all_data = pd.concat((train_df.drop(['SalePrice'], axis=1), test_df))
print(f"Combined data shape: {all_data.shape}")

# Create more meaningful features
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# --- 2d. Handle Missing Values ---
# Fill categorical NAs with 'None' and numerical NAs with the median.
categorical_features = all_data.select_dtypes(exclude=np.number).columns
for col in categorical_features:
    all_data[col] = all_data[col].fillna('None')

numerical_features = all_data.select_dtypes(include=np.number).columns
for col in numerical_features:
    all_data[col] = all_data[col].fillna(all_data[col].median())

# --- 2e. Handle Skewed Features ---
# Log-transform skewed numerical features to make their distributions more normal.
skewed_feats = all_data[numerical_features].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
skewed_feats = skewed_feats.index
all_data[skewed_feats] = np.log1p(all_data[skewed_feats])

# --- 2f. Final Transformations ---
# One-hot encode categorical features.
all_data = pd.get_dummies(all_data, drop_first=True)

# Separate the combined data back into training and testing sets
X = all_data[:train_df.shape[0]]
X_test = all_data[train_df.shape[0]:]

print(f"Processed training features shape: {X.shape}")
print(f"Processed testing features shape: {X_test.shape}")


# --- 3. Model Training ---

# Initialize the XGBoost Regressor with parameters that tend to work well for this competition.
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1 # Use all available CPU cores
)

# Train the model on the training dataset
xgb_model.fit(X, y)

print("\nXGBoost model trained successfully.")

# Calculate RMSE on the training data
train_predictions_log = xgb_model.predict(X)
train_rmse = np.sqrt(mean_squared_error(y, train_predictions_log))
print(f"RMSE on training data (log scale): {train_rmse:.4f}")


# --- 4. Generate Predictions for Submission ---

# Make predictions on the test data
test_predictions_log = xgb_model.predict(X_test)

# The model predicts the log of the price, so we need to convert it back.
final_predictions = np.expm1(test_predictions_log)

# Ensure no negative predictions
final_predictions[final_predictions < 0] = 0


# --- 5. Create Submission File ---

# Create a DataFrame in the format required by Kaggle
submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Save the submission file
submission_df.to_csv('submission_xgboost.csv', index=False)

print("\nSubmission file 'submission_xgboost.csv' created successfully.")
print("You can now upload this file to the Kaggle competition.")
print(submission_df.head())

In [ ]:
# catboost_house_prices.py
# This script trains a CatBoost model on the Ames Housing dataset
# from the Kaggle "House Prices: Advanced Regression Techniques" competition.

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# --- 1. Data Loading and Initial Exploration ---

# Define the correct file paths for the Kaggle environment
TRAIN_PATH = '../input/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '../input/house-prices-advanced-regression-techniques/test.csv'

# Load the training and testing datasets
try:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
except FileNotFoundError:
    print(f"Ensure '{TRAIN_PATH}' and '{TEST_PATH}' are correct.")
    # If running locally, change the paths to 'train.csv' and 'test.csv'
    raise

# Store the 'Id' column from the test set for the final submission file
test_ids = test_df['Id']
# Drop Id columns to avoid treating them as features
train_df = train_df.drop('Id', axis=1)
test_df = test_df.drop('Id', axis=1)

print(f"Original training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

# --- 2. Preprocessing and Feature Engineering ---

# --- 2a. Handle Outliers ---
# The dataset has two known outliers with large GrLivArea but low SalePrice.
# Removing them is a standard practice for this competition.
train_df = train_df.drop(train_df[(train_df['GrLivArea']>4000) & (train_df['SalePrice']<300000)].index)
print(f"Training data shape after removing outliers: {train_df.shape}")


# --- 2b. Target Variable Transformation ---
# The evaluation metric is RMSLE. To optimize for this, we predict the logarithm of SalePrice.
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])
y = train_df['SalePrice']


# --- 2c. Combine Data and Engineer Features ---
# Combine train and test data for consistent preprocessing
all_data = pd.concat((train_df.drop(['SalePrice'], axis=1), test_df))
print(f"Combined data shape: {all_data.shape}")

# Create more meaningful features
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# --- 2d. Handle Missing Values ---
# Fill categorical NAs with 'None' and numerical NAs with the median.
categorical_features = all_data.select_dtypes(exclude=np.number).columns
for col in categorical_features:
    all_data[col] = all_data[col].fillna('None')

numerical_features = all_data.select_dtypes(include=np.number).columns
for col in numerical_features:
    all_data[col] = all_data[col].fillna(all_data[col].median())

# --- 2e. Handle Skewed Features ---
# Log-transform skewed numerical features to make their distributions more normal.
skewed_feats = all_data[numerical_features].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
skewed_feats = skewed_feats.index
all_data[skewed_feats] = np.log1p(all_data[skewed_feats])

# --- 2f. Final Transformations ---
# One-hot encode categorical features.
all_data = pd.get_dummies(all_data, drop_first=True)

# Separate the combined data back into training and testing sets
X = all_data[:train_df.shape[0]]
X_test = all_data[train_df.shape[0]:]

print(f"Processed training features shape: {X.shape}")
print(f"Processed testing features shape: {X_test.shape}")


# --- 3. Model Training ---

# Initialize the CatBoost Regressor with parameters that tend to work well.
# verbose=0 silences the extensive training output.
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    loss_function='RMSE',
    random_seed=42,
    verbose=0
)

# Train the model on the training dataset
cat_model.fit(X, y)

print("\nCatBoost model trained successfully.")

# Calculate RMSE on the training data
train_predictions_log = cat_model.predict(X)
train_rmse = np.sqrt(mean_squared_error(y, train_predictions_log))
print(f"RMSE on training data (log scale): {train_rmse:.4f}")


# --- 4. Generate Predictions for Submission ---

# Make predictions on the test data
test_predictions_log = cat_model.predict(X_test)

# The model predicts the log of the price, so we need to convert it back.
final_predictions = np.expm1(test_predictions_log)

# Ensure no negative predictions
final_predictions[final_predictions < 0] = 0


# --- 5. Create Submission File ---

# Create a DataFrame in the format required by Kaggle
submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Save the submission file
submission_df.to_csv('submission_catboost.csv', index=False)

print("\nSubmission file 'submission_catboost.csv' created successfully.")
print("You can now upload this file to the Kaggle competition.")
print(submission_df.head())


In [ ]:
# This is my final product and final model chosen with best RMSE
# catboost_house_prices.py
# This script trains a CatBoost model on the Ames Housing dataset
# from the Kaggle "House Prices: Advanced Regression Techniques" competition.

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# --- 1. Data Loading and Initial Exploration ---

# Define the correct file paths for the Kaggle environment
TRAIN_PATH = '../input/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '../input/house-prices-advanced-regression-techniques/test.csv'

# Load the training and testing datasets
try:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
except FileNotFoundError:
    print(f"Ensure '{TRAIN_PATH}' and '{TEST_PATH}' are correct.")
    # If running locally, change the paths to 'train.csv' and 'test.csv'
    raise

# Store the 'Id' column from the test set for the final submission file
test_ids = test_df['Id']
# Drop Id columns to avoid treating them as features
train_df = train_df.drop('Id', axis=1)
test_df = test_df.drop('Id', axis=1)

print(f"Original training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

# --- 2. Preprocessing and Feature Engineering ---

# --- 2a. Handle Outliers ---
# The dataset has two known outliers with large GrLivArea but low SalePrice.
# Removing them is a standard practice for this competition.
train_df = train_df.drop(train_df[(train_df['GrLivArea']>4000) & (train_df['SalePrice']<300000)].index)
print(f"Training data shape after removing outliers: {train_df.shape}")


# --- 2b. Target Variable Transformation ---
# Visualize the distribution of SalePrice
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(train_df['SalePrice'], kde=True, bins=30)
plt.title('Distribution of SalePrice (Original)')

# The evaluation metric is RMSLE. To optimize for this, we predict the logarithm of SalePrice.
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])
y = train_df['SalePrice']

# Visualize the new distribution
plt.subplot(1, 2, 2)
sns.histplot(y, kde=True, bins=30)
plt.title('Distribution of SalePrice (Log Transformed)')
plt.show()


# --- 2c. Combine Data and Engineer Features ---
# Combine train and test data for consistent preprocessing
all_data = pd.concat((train_df.drop(['SalePrice'], axis=1), test_df))
print(f"Combined data shape: {all_data.shape}")

# Create more meaningful features
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# --- 2d. Handle Missing Values ---
# Fill categorical NAs with 'None' and numerical NAs with the median.
categorical_features = all_data.select_dtypes(exclude=np.number).columns
for col in categorical_features:
    all_data[col] = all_data[col].fillna('None')

numerical_features = all_data.select_dtypes(include=np.number).columns
for col in numerical_features:
    all_data[col] = all_data[col].fillna(all_data[col].median())

# --- 2e. Handle Skewed Features ---
# Log-transform skewed numerical features to make their distributions more normal.
skewed_feats = all_data[numerical_features].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
skewed_feats = skewed_feats.index
all_data[skewed_feats] = np.log1p(all_data[skewed_feats])

# --- 2f. Final Transformations ---
# One-hot encode categorical features.
all_data = pd.get_dummies(all_data, drop_first=True)

# Separate the combined data back into training and testing sets
X = all_data[:train_df.shape[0]]
X_test = all_data[train_df.shape[0]:]

print(f"Processed training features shape: {X.shape}")
print(f"Processed testing features shape: {X_test.shape}")


# --- 3. Model Training ---

# Initialize the CatBoost Regressor with parameters that tend to work well.
# verbose=0 silences the extensive training output.
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    loss_function='RMSE',
    random_seed=42,
    verbose=0
)

# Train the model on the training dataset
cat_model.fit(X, y)

print("\nCatBoost model trained successfully.")

# Calculate RMSE on the training data
train_predictions_log = cat_model.predict(X)
train_rmse = np.sqrt(mean_squared_error(y, train_predictions_log))
print(f"RMSE on training data (log scale): {train_rmse:.4f}")

# --- 4. Visualizing Feature Impact and Performance ---

# --- 4a. Feature Importance ---
feature_importance = cat_model.get_feature_importance()
feature_imp_df = pd.DataFrame({'feature': X.columns, 'importance': feature_importance})
feature_imp_df = feature_imp_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=feature_imp_df)
plt.title('Top 15 Feature Importances from CatBoost Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# --- 4b. Actual vs. Predicted Prices ---
plt.figure(figsize=(8, 8))
plt.scatter(y, train_predictions_log, alpha=0.5)
plt.plot([min(y), max(y)], [min(y), max(y)], '--', color='red', lw=2) # 45-degree line
plt.xlabel('Actual SalePrice (Log)')
plt.ylabel('Predicted SalePrice (Log)')
plt.title('Actual vs. Predicted Prices on Training Data')
plt.grid(True)
plt.show()


# --- 5. Generate Predictions for Submission ---

# Make predictions on the test data
test_predictions_log = cat_model.predict(X_test)

# The model predicts the log of the price, so we need to convert it back.
final_predictions = np.expm1(test_predictions_log)

# Ensure no negative predictions
final_predictions[final_predictions < 0] = 0


# --- 6. Create Submission File ---

# Create a DataFrame in the format required by Kaggle
submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Save the submission file with a .csv extension
submission_df.to_csv('oshel.bestmodel.csv', index=False)

print("\nSubmission file 'oshel.bestmodel.csv' created successfully.")
print("You can now upload this file to the Kaggle competition.")
print(submission_df.head())